In [ ]:
get_ipython().system('pip install -q transformers datasets torch sqlparse accelerate')

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import sqlparse
from sqlparse.sql import IdentifierList, Identifier, Where, Comparison
from sqlparse.tokens import Keyword, DML
import random
import json
from typing import List, Dict, Tuple
import numpy as np
from dataclasses import dataclass
import re

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

print("Environment ready with imports and seeds set.")

Environment ready with imports and seeds set.


In [ ]:
class SQLQueryGenerator:
  # Create sample queries for training the optimizer. Generator will build model optimization by testing inefficient queries and fixing them.

  def __init__(self):

    self.tables = ['users', 'orders', 'products', 'customers', 'transactions', 'employees', 'departments', 'sales', 'inventory']
    # Common table names

    self.columns = ['id', 'name', 'email', 'created_at', 'updated at', 'status', 'amount', 'quantity', 'price', 'date']
    # Common column names

    self.inefficiency_types = [
        'select_star', # SELECT * instead of specific columns
        'no_index', # Missing WHERE clause that could use index
        'subquery', # Inefficient subquery that could be JOIN
        'multiple_or', # Multiple OR conditions instead of IN
        'no_limit', # Missing LIMIT on large result sets
    ]

  def generate_inefficient_query(self) -> tuple[str, str, str]:

      # Generate an inefficient query and produce an optimized version accompanied by rationale.

      inefficiency = random.choice(self.inefficiency_types)

      table = random.choice(self.tables)

      if inefficiency == 'select_star':
          # SELECT * is inefficient - fetches unnecessary data

          inefficient = f"SELECT * FROM {table} WHERE id > 100"
          cols = random.sample(self.columns[:5], 3)
          optimized = f"SELECT {', '.join(cols)} FROM {table} WHERE id > 100"
          reason = "Replaced SELECT * with specific columns to maximize efficiency and readability"

      elif inefficiency == 'no_index':
          # Missing indexed column in WHERE clause

          col = random.choice(self.columns)
          inefficient = f"SELECT * FROM {table} WHERE {col} LIKE '%value%'"
          optimized = f"SELECT ID, {col} FROM {table} WHERE {col} = 'value'"
          reason = "Used equality insead of LIKE for better indexing"

      elif inefficiency == 'subquery':
        # Inefficient correlated subquery

        table2 = random.choice([t for t in self.tables if t != table])
        inefficient = f"SELECT *FROM {table} WHERE id IN (SELECT user_id FROM {table2})"
        optimized = f"SELECT {table}.* FROM {table} INNER JOIN {table2} ON {table}.id = {table2}.user_id"
        reason = "Replaced subquery with JOIN for better performance"

      elif inefficiency == 'multiple_or':
          # Multiples OR conditions instead of IN clause
          col = random.choice(self.columns)
          vals = [f"'{i}' " for i in range(1, 6)]
          inefficient = f"SELECT * FROM {table} WHERE {col} = {vals[0]} OR {col} = {vals[1]} OR {col} = {vals[2]}"
          optimized = f"SELECT * FROM {table} WHERE {col} IN ({vals[0]}, {vals[1]}, {vals[2].strip()})"
          reason = "Replaced multiple OR conditions with IN clause for conciseness and efficiency"

      else: # no_limit
          # Missing limit returns too much data

          inefficient = f"SELECT * FROM {table} ORDER BY created_at DESC"
          optimized = f"SELECT id, name, created_at FROM {table} ORDER BY created_at DESC LIMIT 100"
          reason = "Added LIMIT clause and specific columns to constrain result set"

      return inefficient, optimized, reason

  def generate_dataset(self, num_samples: int = 1000) -> List[Dict]:
      # Generate a training dataset of query pairs
      # Arguments - num_samples: Number of query optimization examples to generate
      # Returns - List of dictionaries containing query pairs and metadata

      dataset = []
      for i in range(num_samples):
          inefficient, optimized, reason = self.generate_inefficient_query()
          dataset.append({
              'id': i,
              'inefficient_query': inefficient,
              'optimized_query': optimized,
              'optimization_reason': reason,
              'complexity_score': self._calculate_complexity(inefficient)
        })
      return dataset

  def _calculate_complexity(self, quer: str) -> int:
      # Calculate a complexity score for a query based on its features
      # Higher schores = higher complexity

      score = 0
      query_lower = quer.lower()

      score += query_lower.count('join') * 3
      score += query_lower.count('where') * 2
      score += query_lower.count('subquery') * 4
      score += query_lower.count('group by') * 3
      score += query_lower.count('having') * 2
      score += query_lower.count('order by') * 1
      score += len(quer) // 20
        # scoring system for complexity indication

      return score

 # Generate training dataset

print("\n" + "="*70)
print("GENERATING TRAINING DATASET")
print("="*70)

generator = SQLQueryGenerator()
training_data = generator.generate_dataset(num_samples=1000)

print(f"\n Generated {len(training_data)} query optimization examples")
print(f" Average query complexity: {np.mean([d['complexity_score'] for d in training_data]):.2f}")
print(f" Complexity range: {min(d['complexity_score'] for d in training_data)} - {max(d['complexity_score'] for d in training_data)}")
# Show stats of dataset

print("\n" + "-"*70)
print("EXAMPLE QUERY OPTIMIZATION PAIR")
print("-"*70)

example = random.choice(training_data)

print(f"\nInefficient Query:\n{example['inefficient_query']}")
print(f"\nOptimized Query:\n{example['optimized_query']}")
print(f"\nReason: {example['optimization_reason']}")
print(f"Complexity Score: {example['complexity_score']}")
# Show sample from dataset

with open('sql_optimization_dataset.json', 'w') as f:
  json.dump(training_data, f, indent=2)
# Save dataset to file

print("\n Dataset saved to 'sql_optimization_dataset.json'")

print("\n" + "="*70)
print("STAGE 1 COMPLETE - Ready for Stage 2: Model Architecture")
print("="*70)



GENERATING TRAINING DATASET

 Generated 1000 query optimization examples
 Average query complexity: 4.14
 Complexity range: 3 - 6

----------------------------------------------------------------------
EXAMPLE QUERY OPTIMIZATION PAIR
----------------------------------------------------------------------

Inefficient Query:
SELECT * FROM customers WHERE id > 100

Optimized Query:
SELECT name, id, updated at FROM customers WHERE id > 100

Reason: Replaced SELECT * with specific columns to maximize efficiency and readability
Complexity Score: 3

 Dataset saved to 'sql_optimization_dataset.json'

STAGE 1 COMPLETE - Ready for Stage 2: Model Architecture


In [ ]:
def generate_inefficient_query(self) -> tuple[str, str, str]:

  # Generate an inefficient query and produce an optimized version accompanied by rationale.

  inefficiency = random.choice(self.inefficiency_types)

  table = random.choice(self.tables)

  if inefficiency == 'select_star':
    # SELECT * is inefficient - fetches unnecessary data

    inefficient = f"SELECT * FROM {table} WHERE id > 100"
    cols = random.sample(self.columns[:5], 3)
    optimized = f"SELECT {', '.join(cols)} FROM {table} WHERE id > 100"
    reason = "Replaced SELECT * with specific columns to maximize efficiency and readability"

  elif inefficiency == 'no_index':
    # Missing indexed column in WHERE clause

    col = random.choice(self.columns)
    inefficient = f"SELECT * FROM {table} WHERE {col} LIKE '%value%'"
    optimized = f"SELECT ID, {col} FROM {table} WHERE {col} = 'value'"
    reason = "Used equality insead of LIKE for better indexing"

  elif inefficiency == 'subquery':
    # Inefficient correlated subquery

    table2 = random.choice([t for t in self.tables if t != table])
    inefficient = f"SELECT *FROM {table} WHERE id IN (SELECT user_id FROM {table2})"
    optimized = f"SELECT {table}.* FROM {table} INNER JOIN {table2} ON {table}.id = {table2}.user_id"
    reason = "Replaced subquery with JOIN for better performance"

  elif inefficiency == 'multiple_or':
    # Multiples OR conditions instead of IN clause
    col = random.choice(self.columns)
    vals = [f"'{i}' " for i in range(1, 6)]
    inefficient = f"SELECT * FROM [table] WHERE [col] = {vals[0]} OR {col} = {vals[1]} OR {col} = {vals[2]}]"
    reason = "Removed leading wildcard to enable index usage"

  else: no_limit
    # Missing limit returns too much data

  inefficient = f"SELECT * FROM {table} ORDER BY created_at DESC"
  optimized = f"SELECT id, name, created_at FROM {table} ORDER BY created_at DESC LIMIT 100"
  reason = "Added LIMIT clause and specific columns to constrain result set"

  return inefficient, optimized, reason

SQL OPTIMIZER AGENT - MODEL ARCHITECTURE

Building a neural network architecture for the SQL optimizer.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import json
from typing import List, Dict, Tuple, Optional

In [ ]:
class SQLTokenizer:
  def __init__(self, model_name='microsoft/codebert-base'):
    # Initialize with CodeBERT tokenizer - trained on code for SQL

    print("Loading tokenizer (specialized for code understanding)...")
    self.tokenizer = AutoTokenizer.from_pretrained(model_name)

    special_tokens = {
        'additional_special_tokens': [
            '[OPTIMIZE]', # Begin optimization task
            '[EXPLAIN]', # Explanation section
            '[QUERY]' # Query section
        ]
    }

    self.tokenizer.additional_special_tokens(special_tokens)
    print(" Tokenizer loaded with SQL-specific tokens")

def encode_query(self, query: str, max_length: int = 128) -> Dict:
  # convert SQL query text to token IDs with attention masks.

  # Arguments: query:
  # SQL query string
  # max_length: Maximum sequence length

  # Returns:
  # Dictionary with input_ids and attention_mask tensors

  encoding = self.tokenizer(
      query,
      max_length = max_length,
      padding = 'max_length',
      truncation = True,
      return_tensors = 'pt'
  )
  return encoding

def decode_query(self, token_ids: torch.Tensor) -> str:
  # Turn token IDs back into SQL text

  return self.tokenizer.decode(token_ids, skip_special_tokens = True)

In [ ]:
class QueryEncoder(nn.Module):
  # Creating an encoder to turn SQL Queries into dense vector representations so that the model can understand structure, complexity, patterns, syntax, etc.

  def __init__(self, vocab_size: int, embedding_dim: int = 256,
               hidden_dim: int = 512, num_layers: int = 4):
    # Arguments:
    # vocab_size: Size of token vocabulary
    # embedding_dim: Dimension of token embeddings
    # hidden_dim: Size of hidden states in LSTm
    # num_layers: Number of stacked LSTm layers

    super(QueryEncoder, self).__init__()

    self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
    # Embedding layer converts token IDs to dense vectors
    # Each unique token gets a learnable vector representation

    self.lstm = nn.LSTM(
       embedding_dim,
       hidden_dim,
       num_layers = num_layers,
       batch_first = True,
       bidirectional = True,
       dropout = 0.3
)
   # Bidirectional LSTM: reads query forward and backward to help model understand context

    self.projection = nn.Linear(hidden_dim * 2, hidden_dim)
    # Project bidirectional outpouts to hidden_dim
    self.hidden_dim = hidden_dim
    self.num_layers = num_layers

def forward(self, input_ids: torch.Tensor,
            attention_mask: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, Tuple]:

  # Forward pass: encode the input query.

       # Args:
           # input_ids: Token IDs of shape [batch_size, seq_length]
           # attention_mask: Mask for padding tokens

       # Returns:
           # Tuple of (encoded_outputs, hidden_states)

  embedded = self.embedding(input_ids)
  # Convert tokens to embeddings
  # [batch, seq, embed_dim]

  lstm_out, hidden_states = self.lstm(embedded)
  # [batch, seq, hidden * 2]

  encoded = self.projection(lstm_out)
  # Project back to single hidden_dim
  # [batch, seq, hidden_dim]

  return encoded, hidden_states